# 05 — Lite Demo (Real-Time Presentation Version)

This notebook mirrors the full pipeline (Notebooks 01–04) but uses the **lite dataset (2,000 rows)**  
so that every cell runs in seconds — suitable for **live classroom demonstration**.

| Step | Description | Full Version | Lite Version |
|------|-------------|-------------|-------------|
| 1 | Load & Inspect | 39,103 rows | 2,000 rows |
| 2 | TF-IDF + Logistic Regression | ~50s | ~1s |
| 3 | DistilBERT Fine-Tuning | ~15 min (GPU) | ~1 min (GPU) / ~3 min (CPU) |
| 4 | Bias Analysis | ~30s | ~2s |
| 5 | Inference Demo | same | same |

> **Note:** The lite model trained here is for demonstration only.  
> The deployed Flask app uses the full model from `models/fake_news_model/`.

## 0. Imports

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
)

from sklearn.metrics import precision_recall_fscore_support

pd.set_option("display.max_colwidth", 120)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}  |  Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## 1. Load & Inspect the Lite Dataset

In [ ]:
df = pd.read_csv("data/fake_news_lite_clean.csv")

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nClass distribution:")
print(df["label"].value_counts().rename({0: "Real (0)", 1: "Fake (1)"}))
print(f"\nSample rows:")
df[["content", "label"]].head()

---
## 2. TF-IDF + Logistic Regression Baseline

Same pipeline as Notebook 01, but runs in ~1 second on 2,000 rows.

In [ ]:
X = df["content"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")

In [ ]:
tfidf = TfidfVectorizer(
    max_features=20_000,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=2,
    max_df=0.95,
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")
print(f"\nAccuracy:  {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_lr):.4f}")
print(f"\n{classification_report(y_test, y_pred_lr, target_names=['Real', 'Fake'])}")

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm_lr, display_labels=["Real", "Fake"]).plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Confusion Matrix — TF-IDF + LR (Lite)")
plt.tight_layout()
plt.show()

### Top Features

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
coefs = lr.coef_[0]
top_k = 15

top_fake_idx = np.argsort(coefs)[-top_k:][::-1]
top_real_idx = np.argsort(coefs)[:top_k]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].barh(range(top_k), coefs[top_fake_idx][::-1], color="#e74c3c")
axes[0].set_yticks(range(top_k))
axes[0].set_yticklabels(feature_names[top_fake_idx][::-1])
axes[0].set_xlabel("Coefficient")
axes[0].set_title(f"Top {top_k} Tokens → FAKE")

axes[1].barh(range(top_k), np.abs(coefs[top_real_idx])[::-1], color="#2ecc71")
axes[1].set_yticks(range(top_k))
axes[1].set_yticklabels(feature_names[top_real_idx][::-1])
axes[1].set_xlabel("|Coefficient|")
axes[1].set_title(f"Top {top_k} Tokens → REAL")

plt.tight_layout()
plt.show()

---
## 3. DistilBERT Fine-Tuning (Lite)

Same approach as Notebook 02, but with 2,000 rows and 1 epoch.  
The lite model is saved to a **temporary directory** — it does NOT overwrite the full model.

In [ ]:
# Split: 70% train / 15% val / 15% test
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    dist = dict(split["label"].value_counts())
    print(f"{name:>5}: {len(split):>5} rows  |  Real={dist.get(0,0):>4}  Fake={dist.get(1,0):>4}")

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)


class NewsDataset(Dataset):
    """Wraps a DataFrame for the Hugging Face Trainer."""

    def __init__(self, dataframe, tokenizer, max_length):
        self.texts      = dataframe["content"].tolist()
        self.labels     = dataframe["label"].tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }


train_dataset = NewsDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset   = NewsDataset(val_df,   tokenizer, MAX_LENGTH)
test_dataset  = NewsDataset(test_df,  tokenizer, MAX_LENGTH)

print(f"Datasets — Train: {len(train_dataset)}  Val: {len(val_dataset)}  Test: {len(test_dataset)}")

In [ ]:
bert_model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
)

total  = sum(p.numel() for p in bert_model.parameters())
print(f"Parameters: {total:,}")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}


# Save lite model to a temporary directory (NOT overwriting the full model)
LITE_MODEL_DIR = "models/fake_news_model_lite_demo"

training_args = TrainingArguments(
    output_dir=LITE_MODEL_DIR,
    num_train_epochs=1,              # 1 epoch for speed
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print(f"Training on {len(train_dataset)} samples, 1 epoch...")

In [ ]:
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"  Steps: {train_result.global_step}")
print(f"  Loss:  {train_result.training_loss:.4f}")

### Evaluate on Test Set

In [ ]:
test_output = trainer.predict(test_dataset)
test_preds  = np.argmax(test_output.predictions, axis=-1)
test_labels = test_df["label"].values

m = test_output.metrics
print("=" * 50)
print("    LITE MODEL — TEST SET RESULTS")
print("=" * 50)
print(f"  Accuracy  : {m['test_accuracy']:.4f}")
print(f"  Precision : {m['test_precision']:.4f}")
print(f"  Recall    : {m['test_recall']:.4f}")
print(f"  F1 Score  : {m['test_f1']:.4f}")
print("=" * 50)
print(f"\n{classification_report(test_labels, test_preds, target_names=['Real', 'Fake'])}")

In [ ]:
cm_bert = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm_bert, display_labels=["Real", "Fake"]).plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Confusion Matrix — DistilBERT (Lite, 1 epoch)")
plt.tight_layout()
plt.show()

---
## 4. Bias Analysis (Lite)

Quick check for the same dataset artifacts found in Notebook 04.

In [ ]:
real = df[df["label"] == 0]
fake = df[df["label"] == 1]

# Reuters tag check
real_reuters = real["content"].str.contains(r"\(Reuters\)", regex=True).mean()
fake_reuters = fake["content"].str.contains(r"\(Reuters\)", regex=True).mean()

print("Articles containing '(Reuters)':")
print(f"  Real: {real_reuters:.1%}")
print(f"  Fake: {fake_reuters:.1%}")

# Subject overlap
real_subjects = set(real["subject"].unique())
fake_subjects = set(fake["subject"].unique())
overlap = real_subjects & fake_subjects

print(f"\nReal subjects: {real_subjects}")
print(f"Fake subjects: {fake_subjects}")
print(f"Overlap: {overlap if overlap else 'NONE — perfect separation'}")

# Single-feature classifier
reuters_pred = 1 - df["content"].str.contains(r"\(Reuters\)", regex=True).astype(int)
print(f"\nReuters-only classifier accuracy: {accuracy_score(df['label'], reuters_pred):.4f}")

In [ ]:
# Visualization
artifact_data = pd.DataFrame({
    "Class": ["Real", "Fake"],
    "Contains (Reuters) %": [real_reuters * 100, fake_reuters * 100],
})

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(artifact_data["Class"], artifact_data["Contains (Reuters) %"],
              color=["#2ecc71", "#e74c3c"], width=0.5)
ax.set_ylabel("% of articles")
ax.set_title("Reuters Tag Prevalence — Evidence of Data Leakage")
ax.set_ylim(0, 115)
ax.bar_label(bars, fmt="%.1f%%")
plt.tight_layout()
plt.show()

---
## 5. Inference Demo

Use the **full trained model** (from `models/fake_news_model/`) for inference,  
same as the deployed Flask app.

In [ ]:
FULL_MODEL_DIR = "models/fake_news_model"

inf_tokenizer = DistilBertTokenizerFast.from_pretrained(FULL_MODEL_DIR)
inf_model     = DistilBertForSequenceClassification.from_pretrained(FULL_MODEL_DIR)
inf_model.to(DEVICE)
inf_model.eval()

print(f"Full model loaded from: {FULL_MODEL_DIR}")


def predict_text(text: str) -> dict:
    """Predict whether a news article is Fake or Real."""
    inputs = inf_tokenizer(
        text,
        max_length=256,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        logits = inf_model(**inputs).logits

    probs   = F.softmax(logits, dim=-1).squeeze()
    pred_id = torch.argmax(probs).item()
    label   = "Fake" if pred_id == 1 else "Real"

    return {"label": label, "confidence": probs[pred_id].item()}

In [ ]:
demo_texts = [
    "BREAKING: Scientists discover high fat diet actually cures all diseases overnight",
    "WASHINGTON (Reuters) - The Senate voted on Thursday to approve the annual defense policy bill.",
    "You won't BELIEVE what this celebrity did — doctors are SHOCKED!",
    "The Federal Reserve raised interest rates by 0.25% on Wednesday, citing steady economic growth.",
]

print(f"{'Prediction':<10} {'Confidence':>10}  Text")
print("-" * 90)
for t in demo_texts:
    r = predict_text(t)
    print(f"[{r['label']:<4}]      {r['confidence']:>8.2%}    {t[:65]}...")

---
## 6. Cleanup (Optional)

Remove the lite demo model to save disk space.  
The full model in `models/fake_news_model/` is untouched.

In [ ]:
import shutil

if os.path.exists(LITE_MODEL_DIR):
    shutil.rmtree(LITE_MODEL_DIR)
    print(f"Deleted {LITE_MODEL_DIR}/")
else:
    print("Nothing to clean up.")

print(f"Full model still at: models/fake_news_model/  ✓")

## Summary

| Item | Detail |
|------|--------|
| Dataset | `data/fake_news_lite_clean.csv` (2,000 rows, balanced) |
| TF-IDF Baseline | Logistic Regression on 20k features |
| DistilBERT Lite | 1 epoch, batch=16, saved to temp directory |
| Bias Check | Reuters tag & subject leakage confirmed on lite set |
| Inference | Uses full model from `models/fake_news_model/` |
| Purpose | **Live demonstration only** — not for production |